# Voice Analysis with Python — Pitch, Resonance & Attractiveness

This notebook demonstrates how to extract acoustic features from voice recordings that correlate with perceived vocal attractiveness.

**Want instant AI analysis?** Try [RateYourVoice](https://rateyourvoice.com) — 8 AI models score your voice across 9 dimensions in 10 seconds.

## What We'll Cover
1. Fundamental frequency (F0) extraction
2. Harmonic-to-Noise Ratio (HNR)
3. Jitter and Shimmer
4. Formant analysis
5. Spectral features
6. What makes a voice attractive (the research)

In [ ]:
# Install required packages
!pip install -q librosa numpy matplotlib parselmouth

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import parselmouth
from parselmouth.praat import call

## 1. Fundamental Frequency (F0)

F0 is the lowest frequency of your voice — what we perceive as pitch. Research shows:
- Lower F0 in men is rated as more attractive and dominant (Puts et al., 2012)
- CEOs with lower F0 manage larger companies (Mayew et al., 2013)
- F0 range (variation) signals expressiveness and engagement

In [ ]:
def extract_pitch(audio_path):
    """Extract fundamental frequency using Praat via parselmouth."""
    snd = parselmouth.Sound(audio_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]  # Remove unvoiced
    
    return {
        'mean_f0': np.mean(pitch_values),
        'std_f0': np.std(pitch_values),
        'min_f0': np.min(pitch_values),
        'max_f0': np.max(pitch_values),
        'range_f0': np.max(pitch_values) - np.min(pitch_values)
    }

# Example with a sample file
# results = extract_pitch('your_voice.wav')
# print(f"Mean pitch: {results['mean_f0']:.1f} Hz")
# print(f"Pitch range: {results['range_f0']:.1f} Hz")

## 2. Harmonic-to-Noise Ratio (HNR)

HNR measures how 'clean' your voice sounds vs. how breathy/noisy. Higher HNR = smoother, healthier-sounding voice. HNR declines with age but can be improved through vocal exercises.

In [ ]:
def extract_hnr(audio_path):
    """Extract Harmonic-to-Noise Ratio using Praat."""
    snd = parselmouth.Sound(audio_path)
    harmonicity = call(snd, 'To Harmonicity (cc)', 0.01, 75, 0.1, 1.0)
    hnr = call(harmonicity, 'Get mean', 0, 0)
    return hnr

# Typical values:
# > 20 dB: Very clean voice
# 15-20 dB: Normal
# < 15 dB: Breathy or rough

## 3. Jitter & Shimmer

- **Jitter**: Cycle-to-cycle pitch variation (instability)
- **Shimmer**: Cycle-to-cycle amplitude variation

Lower jitter + shimmer = smoother, more attractive voice.

In [ ]:
def extract_jitter_shimmer(audio_path):
    """Extract jitter and shimmer using Praat."""
    snd = parselmouth.Sound(audio_path)
    point_process = call(snd, 'To PointProcess (periodic, cc)', 75, 600)
    
    jitter = call(point_process, 'Get jitter (local)', 0, 0, 0.0001, 0.02, 1.3)
    shimmer = call([snd, point_process], 'Get shimmer (local)', 0, 0, 0.0001, 0.02, 1.3, 1.6)
    
    return {
        'jitter_percent': jitter * 100,
        'shimmer_percent': shimmer * 100
    }

# Healthy voice: jitter < 1%, shimmer < 3%

## The 9 Dimensions of Voice Quality

Research identifies 9 measurable dimensions that determine how a voice is perceived:

| Dimension | Key Acoustic Features |
|-----------|----------------------|
| Depth | F0, lower formants |
| Warmth | HNR, harmonic richness |
| Clarity | Formant distinctness, articulation |
| Confidence | Pitch stability, downward intonation |
| Energy | Dynamic range, speech rate |
| Expressiveness | F0 range, pitch variation |
| Smoothness | Low jitter/shimmer |
| Stability | Pitch steadiness |
| Resonance | Formant spacing, vocal tract characteristics |

## Try the Full Analysis

This notebook covers basic acoustic extraction. For comprehensive AI-powered voice scoring across all 9 dimensions with personalized coaching, try **[RateYourVoice](https://rateyourvoice.com)** — it runs 8 specialized AI models on your voice in 10 seconds.

## References

- Puts et al. (2012). Men's voices as dominance signals. *Evolution and Human Behavior*
- McAleer et al. (2014). How do you say 'hello'? *PLOS ONE*
- Mayew et al. (2013). Voice pitch and CEO compensation. *Evolution and Human Behavior*
- Zuckerman & Driver (1989). What sounds beautiful is good. *Journal of Nonverbal Behavior*